### Homing manual curation after auto detection pipeline


In [1]:
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

from behave_analysis.process.process import Process
from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.analyze.behaviour.homings_escapes.homings import get_Homings
from behave_analysis.analyze.behaviour.homings_escapes.homing_curation_syd_viewer import homing_curation_syd_viewer, save_removed_runs

%matplotlib inline

In [ ]:
# Import experiments
from behave_analysis.database.Experiments.JAL003_ex import JAL3_shelt_17aug, JAL3_mush_21aug, JAL3_flip1_22aug, JAL3_flip2_25aug, JAL3_flip3_29aug, JAL3_flip4_1sept, JAL3_flip5_4sept, JAL3_flip6_7sept

from behave_analysis.database.Experiments.JAL004_ex import JAL4_shelt_17aug, JAL4_mush_18aug, JAL4_flip1_21Aug, JAL4_mush2_22Aug, JAL4_flip3_28aug, JAL4_flip4_3Sept, JAL4_flip5_11Sept, JAL4_flip6_19Sept

from behave_analysis.database.Experiments.JAL005_ex import JAL5_shelt_2Sept, JAL5_barr_5Sept, JAL5_flip1_8Sept, JAL5_flip3_21Sept, JAL5_mush_3oct

from behave_analysis.database.Experiments.JAL006_ex import JAL6_hab_1mar, JAL6_shelt_4mar, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip6_28mar, JAL6_flip7_1apr, JAL6_flip8_5apr


from behave_analysis.database.Experiments.JAL007_ex import JAL7_hab_1mar, JAL7_empty_shelter_5mar, JAL7_flip2_12mar, JAL7_flip3_15mar, JAL7_flip4_19mar, JAL7_flip5_22mar, JAL7_flip7_4apr, JAL7_flip8_9apr, JAL7_flip9_16apr, JAL7_flip10_23apr, JAL7_tiny_30apr

from behave_analysis.database.Experiments.JAL008_ex import (
    JAL8_shelt_22apr,
    JAL8_flip1_25apr,
    JAL8_flip2_29apr,
    JAL8_tiny_3may,
    JAL8_flip3_7may,
    JAL8_flip4_10may,
    JAL8_flip5_14may,
    JAL8_tiny2_21may,
)

experiments_objects = [JAL3_shelt_17aug, JAL3_mush_21aug, JAL3_flip1_22aug, JAL3_flip2_25aug, JAL3_flip3_29aug, JAL3_flip4_1sept, JAL3_flip5_4sept, JAL3_flip6_7sept,
                       JAL4_shelt_17aug, JAL4_mush_18aug, JAL4_flip1_21Aug, JAL4_mush2_22Aug, JAL4_flip3_28aug, JAL4_flip4_3Sept, JAL4_flip5_11Sept, JAL4_flip6_19Sept,
                       JAL5_shelt_2Sept, JAL5_barr_5Sept, JAL5_flip1_8Sept, JAL5_flip3_21Sept, JAL5_mush_3oct,
                       JAL6_hab_1mar, JAL6_shelt_4mar, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip6_28mar, JAL6_flip7_1apr, JAL6_flip8_5apr,
                       JAL7_hab_1mar, JAL7_empty_shelter_5mar, JAL7_flip2_12mar, JAL7_flip3_15mar, JAL7_flip4_19mar, JAL7_flip5_22mar, JAL7_flip7_4apr, JAL7_flip8_9apr, JAL7_flip9_16apr, JAL7_flip10_23apr, JAL7_tiny_30apr,
                       JAL8_shelt_22apr, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_flip5_14may, JAL8_tiny2_21may]

In [259]:
settings = {"homings_speed_threshold": 4.0,  # cm/s, used to find bouts of running that may be homings
            "homings_gap_tolerance": 1,  # frames, used to merge bouts
            "homings_features_initial_window_s": 1.0,  # seconds, used to compute initial features of homings like acceleration and hdir change
            "homing_classification_target_recall": 0.9,  # minimum recall for a gate to be considered valid
            "homings_classification_recall_threshold": 0.9,  # minimum recall for a feature gate to be considered valid
            "homings_classification_precision_threshold": 0.1,  # minimum precision for a feature gate to be considered valid
            "homings_classification_auc_threshold": 0.9,  # or .8, minimum AUC for a feature gate to be considered valid
            "homings_classification_cohens_d_threshold": 1,  # minimum absolute Cohen's d for a feature gate to be considered valid
            "homings_manual_gates": None,
            "redo_compute": False,
            "homings_use_boris": False,
            "homings_curated": False,
            "homings_distance_threshold": 25  # in cm, minimum length to be kept as a homings
            }
from settings.settings_analyze_behave import settings_ab
from settings.settings_overrides import settings_overrides
settings_ab = settings_overrides(settings_ab, settings)

In [222]:
# choose a session and load its data
e = 31

cond_list = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
exp = experiments_objects[e]
session = Process(exp).load_session()

tracking_data = open_tracking_data(session)
video_df_path = os.path.join(session.base_path, session.processed_path, 'full_video_dataframe.csv')
video_df = pl.read_csv(video_df_path)

# open database and check for run with matched settings - if it doesn't exist run it!
homings_dict = get_Homings(settings_ab, session).get_homings(video_df=[], tracking_data=[])  
print(f"Found {len(homings_dict['onset_frames'])} homings in this session")

2026-07-07 13:27:36.706 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:__init__:41 - checking for existing homings results
2026-07-07 13:27:36.785 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['4a0b393b225c443b'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL007\JAL007_barrierflip2_2024_03_12T11_18_26\processed_data\homings\Homing_database.csv
2026-07-07 13:27:36.793 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:get_homings:58 - Homing analysis already done with these settings, loading from database...


Found 108 homings in this session


In [ ]:
# 4. load the homing results and open the syd viewer to manually curate
viewer, removed_runs = homing_curation_syd_viewer(homing_dict=homings_dict, session=session, settings=settings_ab, video_df=video_df, tracking_data=tracking_data, include_manual_events=False, manual_curation=True)
viewer.show()

2026-07-09 12:23:22.780 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels:load_manual_labels:20 - Loaded manual labels
2026-07-09 12:23:22.783 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels:load_manual_labels:21 - Number of homings: 98


In [207]:
removed_runs # remove61

[np.int64(14),
 np.int64(14),
 np.int64(28),
 np.int64(30),
 np.int64(33),
 np.int64(39),
 np.int64(41),
 np.int64(52),
 np.int64(69),
 np.int64(82)]

In [221]:
# 5. save the manual curation to the results dict and add a curated flag to the database entry
save_removed_runs(homings_dict, removed_runs, settings_ab, session)

2026-07-07 13:26:31.157 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['c4f13f4903074df0'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL007\JAL007_empty_shelter_2024_03_05T13_45_47\processed_data\homings\Homing_database.csv


Saving curated homings to Z:\Jasmine_Laurence\Experimental_Data\JAL007\JAL007_empty_shelter_2024_03_05T13_45_47\processed_data\homings\homings_c4f13f4903074df0_results.npy and database entry to Z:\Jasmine_Laurence\Experimental_Data\JAL007\JAL007_empty_shelter_2024_03_05T13_45_47\processed_data\homings\Homing_database.csv
